# ACUHIT Sentinel — 01: Veri Keşif Analizi
**Amaç:** Birleştirmeden önce her veri setini anlamak.



In [1]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.2f}'.format)

print(' Kütüphaneler yüklendi')

 Kütüphaneler yüklendi


---
## 📁 BÖLÜM 1: Klasör Yapısını Gör

In [4]:
BASE_PATH = r"C:\Users\Mutlu\Desktop\ACUHIT\data"

print('=== KLASÖR YAPISI ===')
for root, dirs, files in os.walk(BASE_PATH):
    # Sadece CSV dosyalarını say
    csv_files = [f for f in files if f.endswith('.csv')]
    level = root.replace(BASE_PATH, '').count(os.sep)
    indent = '  ' * level
    folder_name = os.path.basename(root)
    if csv_files:
        print(f'{indent}📂 {folder_name}/ → {len(csv_files)} CSV')
        for f in csv_files[:3]:  # İlk 3'ü göster
            size = os.path.getsize(os.path.join(root, f)) / 1024
            print(f'{indent}   📄 {f} ({size:.0f} KB)')
        if len(csv_files) > 3:
            print(f'{indent}   ... ve {len(csv_files)-3} dosya daha')
    elif not csv_files and level <= 2:
        print(f'{indent}📂 {folder_name}/')

=== KLASÖR YAPISI ===
📂 data/
  📂 Cancer - Data/
    📂 Cancer_Anadata/ → 1 CSV
       📄 ca_anadata_0001.csv (171646 KB)
    📂 Cancer_Lab/ → 4 CSV
       📄 ca_lab_0001.csv (66023 KB)
       📄 ca_lab_0002.csv (66769 KB)
       📄 ca_lab_0003.csv (67451 KB)
       ... ve 1 dosya daha
    📂 Cancer_Recete/ → 1 CSV
       📄 ca_recete_0001.csv (17900 KB)
  📂 Check-Up - Data/
    📂 Check_Up_Anadata/ → 6 CSV
       📄 Anadata_0001.csv (962674 KB)
       📄 Anadata_0002.csv (958349 KB)
       📄 Anadata_0003.csv (943315 KB)
       ... ve 3 dosya daha
    📂 Check_Up_Lab/ → 72 CSV
       📄 lab_0001.csv (70664 KB)
       📄 lab_0002.csv (70345 KB)
       📄 lab_0003.csv (70225 KB)
       ... ve 69 dosya daha
    📂 Check_Up_Recete/ → 4 CSV
       📄 prescriptions_0001.csv (98525 KB)
       📄 prescriptions_0002.csv (102482 KB)
       📄 prescriptions_0003.csv (105711 KB)
       ... ve 1 dosya daha
  📂 Ex - Data/
    📂 Ex_Anadata/ → 1 CSV
       📄 ex_anadata_0001.csv (39483 KB)
    📂 Ex_Lab/ → 2 CSV
       📄 

In [5]:
# Tüm CSV dosyalarını bir sözlükte topla
all_files = {}
for root, dirs, files in os.walk(BASE_PATH):
    for f in files:
        if f.endswith('.csv'):
            full_path = os.path.join(root, f)
            # Kategorik etiket çıkar (Cancer/CheckUp/Ex + Ana/Lab/Recete)
            parts = root.replace(BASE_PATH, '').split(os.sep)
            parts = [p for p in parts if p]
            label = ' > '.join(parts)
            if label not in all_files:
                all_files[label] = []
            all_files[label].append(full_path)

print('=== DOSYA ÖZETI ===')
for label, paths in sorted(all_files.items()):
    total_size = sum(os.path.getsize(p) for p in paths) / (1024*1024)
    print(f'  {label:<45} → {len(paths):>3} dosya  ({total_size:.1f} MB)')

=== DOSYA ÖZETI ===
  Cancer - Data > Cancer_Anadata                →   1 dosya  (167.6 MB)
  Cancer - Data > Cancer_Lab                    →   4 dosya  (219.4 MB)
  Cancer - Data > Cancer_Recete                 →   1 dosya  (17.5 MB)
  Check-Up - Data > Check_Up_Anadata            →   6 dosya  (5329.3 MB)
  Check-Up - Data > Check_Up_Lab                →  72 dosya  (4854.5 MB)
  Check-Up - Data > Check_Up_Recete             →   4 dosya  (357.3 MB)
  Ex - Data > Ex_Anadata                        →   1 dosya  (38.6 MB)
  Ex - Data > Ex_Lab                            →   2 dosya  (101.0 MB)
  Ex - Data > Ex_Recete                         →   1 dosya  (2.9 MB)


---
## 🔬 BÖLÜM 2: Tek CSV Aç ve Yapıyı Anla


In [6]:
def load_one(path, nrows=None):
    """Tek CSV'yi akıllıca yükler. sep otomatik, encoding otomatik."""
    try:
        df = pd.read_csv(path, sep=None, engine='python',
                         encoding='utf-8', on_bad_lines='skip',
                         nrows=nrows)
    except UnicodeDecodeError:
        df = pd.read_csv(path, sep=None, engine='python',
                         encoding='latin-1', on_bad_lines='skip',
                         nrows=nrows)
    return df

print('✅ load_one() hazır')

✅ load_one() hazır


In [8]:

SAMPLE_FILES = {
    'cancer_ana':  r"C:\Users\Mutlu\Desktop\ACUHIT\data\Cancer - Data\Cancer_Anadata\ca_anadata_0001.csv",
    'cancer_lab':  r"C:\Users\Mutlu\Desktop\ACUHIT\data\Cancer - Data\Cancer_Lab\ca_lab_0001.csv",
    'cancer_rec':  r"C:\Users\Mutlu\Desktop\ACUHIT\data\Cancer - Data\Cancer_Recete\ca_recete_0001.csv",
    'checkup_ana': r"C:\Users\Mutlu\Desktop\ACUHIT\data\Check-Up - Data\Check_Up_Anadata\Anadata_0001.csv",
    'checkup_lab': r"C:\Users\Mutlu\Desktop\ACUHIT\data\Check-Up - Data\Check_Up_Lab\lab_0001.csv",
    'checkup_rec': r"C:\Users\Mutlu\Desktop\ACUHIT\data\Check-Up - Data\Check_Up_Recete\prescriptions_0001.csv",
    'ex_ana':      r"C:\Users\Mutlu\Desktop\ACUHIT\data\Ex - Data\Ex_Anadata\ex_anadata_0001.csv",
    'ex_lab':      r"C:\Users\Mutlu\Desktop\ACUHIT\data\Ex - Data\Ex_Lab\ex_lab_0001.csv",
    'ex_rec':      r"C:\Users\Mutlu\Desktop\ACUHIT\data\Ex - Data\Ex_Recete\ex_recete_0001.csv",
}

CHECKUP_LAB_PATH = r"C:\Users\Mutlu\Desktop\ACUHIT\data\Check-Up - Data\Check_Up_Lab"

# Var olan dosyaları yükle
dfs = {}
for key, path in SAMPLE_FILES.items():
    if os.path.exists(path):
        df = load_one(path, nrows=500)  # İlk 500 satır yeterli
        dfs[key] = df
        print(f'✅ {key:<15} → {df.shape[0]} satır, {df.shape[1]} sütun | {path.split("/")[-1]}')
    else:
        print(f'⚠️  {key:<15} → DOSYA BULUNAMADI: {path}')

✅ cancer_ana      → 500 satır, 59 sütun | C:\Users\Mutlu\Desktop\ACUHIT\data\Cancer - Data\Cancer_Anadata\ca_anadata_0001.csv
✅ cancer_lab      → 500 satır, 7 sütun | C:\Users\Mutlu\Desktop\ACUHIT\data\Cancer - Data\Cancer_Lab\ca_lab_0001.csv
✅ cancer_rec      → 500 satır, 9 sütun | C:\Users\Mutlu\Desktop\ACUHIT\data\Cancer - Data\Cancer_Recete\ca_recete_0001.csv
✅ checkup_ana     → 500 satır, 61 sütun | C:\Users\Mutlu\Desktop\ACUHIT\data\Check-Up - Data\Check_Up_Anadata\Anadata_0001.csv
✅ checkup_lab     → 500 satır, 7 sütun | C:\Users\Mutlu\Desktop\ACUHIT\data\Check-Up - Data\Check_Up_Lab\lab_0001.csv
✅ checkup_rec     → 500 satır, 9 sütun | C:\Users\Mutlu\Desktop\ACUHIT\data\Check-Up - Data\Check_Up_Recete\prescriptions_0001.csv
✅ ex_ana          → 500 satır, 59 sütun | C:\Users\Mutlu\Desktop\ACUHIT\data\Ex - Data\Ex_Anadata\ex_anadata_0001.csv
✅ ex_lab          → 500 satır, 7 sütun | C:\Users\Mutlu\Desktop\ACUHIT\data\Ex - Data\Ex_Lab\ex_lab_0001.csv
✅ ex_rec          → 500 satır, 

---
## 🗂️ BÖLÜM 3: Her Tablonun Kolon Yapısı

In [9]:
def analyze_columns(df, name):
    """Bir DataFrame'in tüm sütunlarını analiz eder."""
    print(f'\n{"="*70}')
    print(f'  {name.upper()}  ({df.shape[0]} satır × {df.shape[1]} sütun)')
    print(f'{"="*70}')
    
    rows = []
    for col in df.columns:
        null_pct  = df[col].isnull().mean() * 100
        n_unique  = df[col].nunique()
        dtype_str = str(df[col].dtype)
        
        # Örnek değer
        sample = df[col].dropna().iloc[0] if df[col].notna().any() else 'BOŞ'
        sample = str(sample)[:45]
        
        # Durum bayrağı
        if null_pct == 100:
            flag = '❌ TAM BOŞ'
        elif null_pct >= 80:
            flag = '⚠️  ÇOĞU BOŞ'
        elif null_pct >= 30:
            flag = '🟡 KISMİ BOŞ'
        else:
            flag = '✅ DOLU'
        
        rows.append({
            'Sütun': col,
            'Flag': flag,
            'Null%': f'{null_pct:.0f}%',
            'Unique': n_unique,
            'Dtype': dtype_str,
            'Örnek': sample
        })
    
    result = pd.DataFrame(rows)
    print(result.to_string(index=False))
    
    # Özet
    tam_bos  = sum(1 for r in rows if '❌' in r['Flag'])
    cok_bos  = sum(1 for r in rows if '⚠️' in r['Flag'])
    kismi    = sum(1 for r in rows if '🟡' in r['Flag'])
    dolu     = sum(1 for r in rows if '✅' in r['Flag'])
    print(f'\n  Özet: ✅ {dolu} dolu | 🟡 {kismi} kısmi | ⚠️ {cok_bos} çoğu boş | ❌ {tam_bos} tam boş')

print('✅ analyze_columns() hazır')

✅ analyze_columns() hazır


In [10]:
# Her yüklenen tabloyu analiz et
for key, df in dfs.items():
    analyze_columns(df, key)


  CANCER_ANA  (500 satır × 59 sütun)
                      Sütun         Flag Null%  Unique   Dtype                                         Örnek
                   HASTA_ID       ✅ DOLU    0%       6  object                                   ANON_243337
                 SQ_EPISODE       ✅ DOLU    0%     229   int64                                      20654374
              EPISODE_TARIH       ✅ DOLU    0%     284  object                           2016-10-12 11:35:34
        TOPLAM_GELIS_SAYISI       ✅ DOLU    0%       7   int64                                           138
               GELIS_SAYISI       ✅ DOLU    0%       7   int64                                            81
ILK_TANI_SON_TANI_GUN_FARKI       ✅ DOLU    1%       6 float64                                        6761.0
                  TANI_YASI       ✅ DOLU    0%      22   int64                                            63
                   CINSIYET       ✅ DOLU    0%       2  object                            

---
## 🔑 BÖLÜM 4: Birleştirme Anahtarlarını Kontrol Et

In [17]:
print('=== BİRLEŞTİRME ANAHTARI ANALİZİ ===\n')

# Olası anahtar adları
KEY_CANDIDATES = [
    'HASTANO', 'HASTA_NO', 'PATIENT_ID', 'PatientID', 'HASTA_ID',
    'SQ_EPISODE', 'EPISODE_ID', 'EPISODENO',
    'RF_EPISODE', 'EPISODE_NO',
]

for key, df in dfs.items():
    print(f'--- {key} ---')
    cols_upper = {c.upper(): c for c in df.columns}
    
    found = []
    for candidate in KEY_CANDIDATES:
        if candidate.upper() in cols_upper:
            real_col = cols_upper[candidate.upper()]
            n_unique = df[real_col].nunique()
            n_total  = len(df)
            is_unique = n_unique == n_total
            found.append((real_col, n_unique, n_total, is_unique))
    
    if found:
        for col, nu, nt, isu in found:
            unique_flag = '🔑 UNIQUE' if isu else f'↩️  TEKRAR ({nt/nu:.1f}x)'
            print(f'  {col:<20} → {nu:>6} unique / {nt:>6} toplam  {unique_flag}')
    else:
        print(f'  ⚠️  Bilinen anahtar sütunu bulunamadı!')
        print(f'  Mevcut sütunlar: {list(df.columns[:8])}...')
    print()

=== BİRLEŞTİRME ANAHTARI ANALİZİ ===

--- cancer_ana ---
  HASTA_ID             →      6 unique /    500 toplam  ↩️  TEKRAR (83.3x)
  SQ_EPISODE           →    229 unique /    500 toplam  ↩️  TEKRAR (2.2x)

--- cancer_lab ---
  HASTA_ID             →    189 unique /    500 toplam  ↩️  TEKRAR (2.6x)

--- cancer_rec ---
  HASTA_ID             →    226 unique /    500 toplam  ↩️  TEKRAR (2.2x)
  RF_EPISODE           →    397 unique /    500 toplam  ↩️  TEKRAR (1.3x)

--- checkup_ana ---
  HASTA_ID             →     11 unique /    500 toplam  ↩️  TEKRAR (45.5x)
  SQ_EPISODE           →    203 unique /    500 toplam  ↩️  TEKRAR (2.5x)

--- checkup_lab ---
  HASTA_ID             →      3 unique /    500 toplam  ↩️  TEKRAR (166.7x)

--- checkup_rec ---
  HASTA_ID             →    410 unique /    500 toplam  ↩️  TEKRAR (1.2x)
  RF_EPISODE           →    420 unique /    500 toplam  ↩️  TEKRAR (1.2x)

--- ex_ana ---
  HASTA_ID             →     58 unique /    500 toplam  ↩️  TEKRAR (8.6x)
  SQ_E

---
## 📅 BÖLÜM 5: Tarih Sütunlarını Kontrol Et

In [20]:
EXCEL_EPOCH = datetime(1899, 12, 30)

def detect_date_format(series):
    sample = series.dropna().head(5)
    results = []
    for val in sample:
        val_str = str(val).strip()
        for fmt in ['%Y-%m-%d %H:%M:%S', '%Y-%m-%d']:
            try:
                dt = datetime.strptime(val_str[:19], fmt)
                results.append(f'STRING ({fmt}) → {dt.strftime("%Y-%m-%d")} ({val_str[:20]})')
                break
            except:
                continue
        else:
            results.append(f'BİLİNMİYOR → {val_str[:20]}')
    return results

print('=== TARİH SÜTUNLARI ===\n')
DATE_KEYWORDS = ['TARIH', 'DATE', 'TARİH', 'TARİHİ']

for key, df in dfs.items():
    date_cols = [c for c in df.columns 
                 if any(kw in c.upper() for kw in DATE_KEYWORDS)]
    if date_cols:
        print(f'--- {key} ---')
        for col in date_cols:
            fmt_results = detect_date_format(df[col])
            print(f'  {col}:')
            for r in fmt_results[:2]:
                print(f'    {r}')
        print()

=== TARİH SÜTUNLARI ===

--- cancer_ana ---
  EPISODE_TARIH:
    STRING (%Y-%m-%d %H:%M:%S) → 2016-10-12 (2016-10-12 11:35:34)
    STRING (%Y-%m-%d %H:%M:%S) → 2016-10-12 (2016-10-12 11:35:34)
  TANITARIH:
    STRING (%Y-%m-%d %H:%M:%S) → 2016-10-12 (2016-10-12 11:35:33)
    STRING (%Y-%m-%d %H:%M:%S) → 2016-10-07 (2016-10-07 12:00:49)
  MIN_TANI_TARIH:
    STRING (%Y-%m-%d %H:%M:%S) → 2007-07-19 (2007-07-19 11:09:29)
    STRING (%Y-%m-%d %H:%M:%S) → 2007-07-19 (2007-07-19 11:09:29)
  MAX_TANI_TARIH:
    STRING (%Y-%m-%d %H:%M:%S) → 2026-01-21 (2026-01-21 10:19:49)
    STRING (%Y-%m-%d %H:%M:%S) → 2026-01-21 (2026-01-21 10:19:49)

--- cancer_lab ---
  REP_DATE:
    STRING (%Y-%m-%d %H:%M:%S) → 2018-11-27 (2018-11-27 08:55:52)
    STRING (%Y-%m-%d %H:%M:%S) → 2019-11-23 (2019-11-23 16:27:34)

--- cancer_rec ---
  RECETE_TARIH:
    STRING (%Y-%m-%d %H:%M:%S) → 2024-09-18 (2024-09-18 08:31:58)
    STRING (%Y-%m-%d %H:%M:%S) → 2024-08-24 (2024-08-24 10:06:04)

--- checkup_ana ---
  EPISODE

---
## 🏥 BÖLÜM 6: Kohortlar Arası Karşılaştırma
Aynı tür tablolarda (Ana/Lab/Recete) sütunlar aynı mı?

In [21]:
print('=== KOLON KARŞILAŞTIRMASI: ANA VERİ ===\n')

ana_keys = [k for k in dfs if 'ana' in k]
if len(ana_keys) >= 2:
    all_cols = set()
    for k in ana_keys:
        all_cols.update(dfs[k].columns)
    
    print(f'  {"Sütun":<40}', end='')
    for k in ana_keys:
        print(f'  {k:<15}', end='')
    print()
    print('-' * (40 + len(ana_keys) * 17))
    
    only_in_some = []
    for col in sorted(all_cols):
        presence = []
        for k in ana_keys:
            presence.append('✅' if col in dfs[k].columns else '❌')
        
        if '❌' in presence:  # Sadece farklıları göster
            print(f'  {col:<40}', end='')
            for p in presence:
                print(f'  {p:<15}', end='')
            print()
            only_in_some.append(col)
    
    if not only_in_some:
        print('  ✅ Tüm ana veri tablolarında AYNI kolonlar var — birleştirme kolay!')
    else:
        print(f'\n  ⚠️  {len(only_in_some)} sütun kohortlar arasında farklı!')
else:
    print('  Karşılaştırma için birden fazla ana veri tablosu yükleyin.')

=== KOLON KARŞILAŞTIRMASI: ANA VERİ ===

  Sütun                                     cancer_ana       checkup_ana      ex_ana         
-------------------------------------------------------------------------------------------
  BASLANGIC_ZAMANI                          ❌                ✅                ❌              
  RF_EPISODE2                               ❌                ✅                ❌              

  ⚠️  2 sütun kohortlar arasında farklı!


In [22]:
print('=== KOLON KARŞILAŞTIRMASI: LAB VERİSİ ===\n')

lab_keys = [k for k in dfs if 'lab' in k]
if len(lab_keys) >= 2:
    all_cols = set()
    for k in lab_keys:
        all_cols.update(dfs[k].columns)
    
    print(f'  {"Sütun":<30}', end='')
    for k in lab_keys:
        print(f'  {k:<15}', end='')
    print()
    
    for col in sorted(all_cols):
        presence = ['✅' if col in dfs[k].columns else '❌' for k in lab_keys]
        if '❌' in presence:
            print(f'  {col:<30}', end='')
            for p in presence:
                print(f'  {p:<15}', end='')
            print()
    
    common = set.intersection(*[set(dfs[k].columns) for k in lab_keys])
    print(f'\n  Ortak kolon sayısı: {len(common)}')
    print(f'  Ortak kolonlar: {sorted(common)}')
else:
    print('  Karşılaştırma için birden fazla lab tablosu yükleyin.')

=== KOLON KARŞILAŞTIRMASI: LAB VERİSİ ===

  Sütun                           cancer_lab       checkup_lab      ex_lab         

  Ortak kolon sayısı: 7
  Ortak kolonlar: ['HASTA_ID', 'REFMAX', 'REFMIN', 'REP_DATE', 'RESULT', 'SUB_CODE', 'UNIT']


---
## 🧬 BÖLÜM 7: Kanser Verisine Özel Bakış
Cancer kohortunda onkoloji'ye özgü kolonlar var mı?

In [23]:
print('=== KANSER VERİSİ: ONKOLOJİ ÖZGÜ SÜTUNLAR ===\n')

ONCO_KEYWORDS = [
    'tumor', 'tümör', 'stage', 'evre', 'metastaz', 'kemo',
    'radyo', 'biopsi', 'biyopsi', 'CEA', 'CA125', 'CA19',
    'PSA', 'histoloji', 'patoloji', 'survi', 'relaps'
]


if 'cancer_ana' not in dfs:
    cancer_ana_path = r"C:\Users\Mutlu\Desktop\ACUHIT\data\Cancer - Data\Cancer_Anadata\ca_anadata_0001.csv"
    dfs['cancer_ana'] = load_one(cancer_ana_path, nrows=500)
    print(f'✅ cancer_ana yüklendi')

if 'cancer_ana' in dfs:
    df = dfs['cancer_ana']
    print(f'Cancer Ana: {df.shape}')
    
    # Onkoloji keyword'leri içeren sütunlar
    onco_cols = [c for c in df.columns 
                 if any(kw.lower() in c.lower() for kw in ONCO_KEYWORDS)]
    
    if onco_cols:
        print(f'\n  Onkolojiye özgü sütunlar ({len(onco_cols)}):')
        for col in onco_cols:
            null_pct = df[col].isnull().mean() * 100
            sample = str(df[col].dropna().iloc[0])[:40] if df[col].notna().any() else 'BOŞ'
            print(f'    {col:<35} null=%{null_pct:.0f}  örnek: {sample}')
    else:
        print('  Onkolojiye özgü sütun bulunamadi')
        print('  (Kolonlar diğer kohortlarla aynı olabilir)')
    
    # Servis/bölüm dağılımı
    servis_col = next((c for c in df.columns 
                       if 'SERVIS' in c.upper() or 'BOLUM' in c.upper()), None)
    if servis_col:
        print(f'\n  Bolum dagilimi ({servis_col}):')
        print(df[servis_col].value_counts().head(10).to_string())
    
    # ICD kodları — TANIKODU sütunundan
    icd_col = next((c for c in df.columns 
                    if 'TANIKODU' in c.upper() or ('TANI' in c.upper() and 'KOD' in c.upper())), None)
    if icd_col:
        print(f'\n  ICD dagilimi - {icd_col} (ilk harf = kategori):')
        icd_cats = df[icd_col].dropna().str[0].value_counts()
        icd_map = {
            'C': 'Kanser',
            'D': 'Benign Tm / Kan',
            'E': 'Endokrin',
            'F': 'Psikiyatri',
            'G': 'Noroloji',
            'I': 'Kardiyovaskuler',
            'J': 'Solunum',
            'K': 'Sindirim',
            'M': 'Kas-Iskelet',
            'N': 'Urogenital',
            'R': 'Semptom',
            'Z': 'Kontrol',
        }
        for cat, count in icd_cats.head(12).items():
            label = icd_map.get(cat, 'Diger')
            bar = '█' * int(count / icd_cats.max() * 20)
            print(f'    {cat} ({label:<22}): {count:>4}  {bar}')
    
    # TUM_EPS_TANILAR — tüm tanı geçmişi
    if 'TUM_EPS_TANILAR' in df.columns:
        print(f'\n  TUM_EPS_TANILAR ornek:')
        for i, val in enumerate(df['TUM_EPS_TANILAR'].dropna().head(3)):
            print(f'    [{i+1}] {str(val)[:80]}')
        
        # C kodu (kanser) geçen satır oranı
        c_rate = df['TUM_EPS_TANILAR'].str.contains(r'\bC\d', na=False).mean() * 100
        print(f'\n  Kanser ICD (C*) gecmisi olan hasta orani: %{c_rate:.0f}')

=== KANSER VERİSİ: ONKOLOJİ ÖZGÜ SÜTUNLAR ===

Cancer Ana: (500, 59)
  Onkolojiye özgü sütun bulunamadi
  (Kolonlar diğer kohortlarla aynı olabilir)

  Bolum dagilimi (SERVISADI):
SERVISADI
Check Up ve Sağlıklı Yaşam     84
İç Hastalıkları                76
Deri Hastalıkları              62
Kulak Burun Boğaz              51
Kardiyoloji                    39
Kadın Hastalıkları ve Doğum    33
Ortopedi ve Travmatoloji       28
Göğüs Hastalıkları             20
Acil Servis                    17
Gastroenteroloji               14

  ICD dagilimi - TANIKODU (ilk harf = kategori):
    J (Solunum               ):   72  ████████████████████
    E (Endokrin              ):   72  ████████████████████
    L (Diger                 ):   58  ████████████████
    M (Kas-Iskelet           ):   49  █████████████
    R (Semptom               ):   46  ████████████
    D (Benign Tm / Kan       ):   43  ███████████
    N (Urogenital            ):   42  ███████████
    I (Kardiyovaskuler       ):   30  ██████

---
## 📊 BÖLÜM 8: Check-Up Lab — 72 Dosya Yapısı

In [24]:
# BÖLÜM 8 — Check-Up Lab 72 dosya yapısı
CHECKUP_LAB_PATH = r"C:\Users\Mutlu\Desktop\ACUHIT\data\Check-Up - Data\Check_Up_Lab"

import os
csv_files = sorted([f for f in os.listdir(CHECKUP_LAB_PATH) if f.endswith('.csv')])
print(f'Check-Up Lab klasöründe {len(csv_files)} CSV dosyası var')
print(f'İlk 5 dosya: {csv_files[:5]}')
print(f'Son 5 dosya: {csv_files[-5:]}')

print(f'\nİlk 3 dosya satır sayıları:')
for fname in csv_files[:3]:
    fpath = os.path.join(CHECKUP_LAB_PATH, fname)
    df_tmp = load_one(fpath)
    size_mb = os.path.getsize(fpath) / (1024*1024)
    print(f'  {fname:<35} → {len(df_tmp):>7} satır  ({size_mb:.1f} MB)')

first_rows = len(load_one(os.path.join(CHECKUP_LAB_PATH, csv_files[0])))
total_est  = first_rows * len(csv_files)
print(f'\nTahmini toplam satır: ~{total_est:,}')

Check-Up Lab klasöründe 72 CSV dosyası var
İlk 5 dosya: ['lab_0001.csv', 'lab_0002.csv', 'lab_0003.csv', 'lab_0004.csv', 'lab_0005.csv']
Son 5 dosya: ['lab_0068.csv', 'lab_0069.csv', 'lab_0070.csv', 'lab_0071.csv', 'lab_0072.csv']

İlk 3 dosya satır sayıları:
  lab_0001.csv                        →  999999 satır  (69.0 MB)
  lab_0002.csv                        →  999999 satır  (68.7 MB)
  lab_0003.csv                        →  999999 satır  (68.6 MB)

Tahmini toplam satır: ~71,999,928


---
## 🧪 BÖLÜM 9: Lab Verisinin Derinliği
Hangi testler var? Kaç benzersiz test? Referans değerleri dolu mu?

In [25]:
print('=== LAB VERİSİ DERİNLİK ANALİZİ ===\n')

for key in [k for k in dfs if 'lab' in k]:
    df = dfs[key]
    print(f'--- {key} ({len(df)} satır) ---')
    
    # Test adı kolonu bul
    test_col = next((c for c in df.columns 
                     if any(kw in c.upper() for kw in
                            ['SUB_CODE','TEST_ADI','TEST_NAME','TESTADI',
                             'ANALYTE','PARAM'])), None)
    
    if test_col:
        n_tests = df[test_col].nunique()
        print(f'  Benzersiz test sayısı: {n_tests}')
        print(f'  En sık 15 test:')
        print(df[test_col].value_counts().head(15).to_string())
    
    # Sonuç kolonu
    sonuc_col = next((c for c in df.columns 
                      if c.upper() in ['SONUC','RESULT','SONUÇ','VALUE']), None)
    if sonuc_col:
        num_vals = pd.to_numeric(df[sonuc_col], errors='coerce')
        print(f'\n  Sonuç numerik oran: {num_vals.notna().mean()*100:.0f}%')
        print(f'  Sonuç istatistik: min={num_vals.min():.2f}, max={num_vals.max():.2f}, mean={num_vals.mean():.2f}')
    
    # Referans değerleri
    ref_cols = [c for c in df.columns if 'REF' in c.upper() or 'NORM' in c.upper()]
    if ref_cols:
        print(f'\n  Referans sütunları: {ref_cols}')
        for rc in ref_cols:
            null_pct = df[rc].isnull().mean() * 100
            print(f'    {rc}: %{null_pct:.0f} boş')
    print()

=== LAB VERİSİ DERİNLİK ANALİZİ ===

--- cancer_lab (500 satır) ---
  Benzersiz test sayısı: 6
  En sık 15 test:
SUB_CODE
 pH (İdrar analizi)                    223
Üre Azotu (BUN)                        182
 Hemoglobin                             87
Toxoplasma Antikoru, IgG                 4
Cytomegalovirus (CMV) Antikoru, IgG      3
Kreatin Kinaz-MB (CK-MB), Kütle          1

  Sonuç numerik oran: 99%
  Sonuç istatistik: min=0.40, max=32.00, mean=10.39

  Referans sütunları: ['REFMIN', 'REFMAX']
    REFMIN: %2 boş
    REFMAX: %1 boş

--- checkup_lab (500 satır) ---
  Benzersiz test sayısı: 128
  En sık 15 test:
SUB_CODE
Kreatinin, Serum                                       11
Alanin Aminotransferaz (ALT)                            9
eGFR                                                    9
Potasyum (K)                                            8
Üre Azotu (BUN)                                         8
Sodyum (Na)                                             8
Üre                   

---
## 💊 BÖLÜM 10: Reçete Verisinin Zenginliği

In [26]:
print('=== REÇETE VERİSİ ANALİZİ ===\n')

NSAID_KEYWORDS = ['naproksen','diklofenak','ibuprofen','dikloron',
                  'voltaren','arveles','majezik','celebrex']
CHEMO_KEYWORDS = ['sisplatin','oxaliplatin','docetaxel','paclitaxel',
                  'fluorourasil','kapesitabin','bevacizumab','trastuzumab']

for key in [k for k in dfs if 'rec' in k]:
    df = dfs[key]
    print(f'--- {key} ({len(df)} satır) ---')
    
    # İlaç adı kolonu bul
    ilac_col = next((c for c in df.columns 
                     if any(kw in c.upper() for kw in
                            ['İLAÇ','ILAC','DRUG','MEDICINE','ADI'])), None)
    
    if ilac_col:
        print(f'  İlaç adı kolonu: {ilac_col}')
        print(f'  Benzersiz ilaç sayısı: {df[ilac_col].nunique()}')
        print(f'  En sık 10 ilaç:')
        print(df[ilac_col].value_counts().head(10).to_string())
        
        # NSAİİ var mı?
        ilac_lower = df[ilac_col].str.lower().fillna('')
        nsaid_count = sum(ilac_lower.str.contains(kw).sum() for kw in NSAID_KEYWORDS)
        chemo_count = sum(ilac_lower.str.contains(kw).sum() for kw in CHEMO_KEYWORDS)
        
        print(f'\n  NSAİİ grubundan ilaç: {nsaid_count} kayıt')
        print(f'  Kemoterapi grubundan ilaç: {chemo_count} kayıt')
    
    # Diğer kolonlar
    print(f'\n  Tüm kolonlar: {list(df.columns)}')
    print()

=== REÇETE VERİSİ ANALİZİ ===

--- cancer_rec (500 satır) ---
  İlaç adı kolonu: İlaç Adı
  Benzersiz ilaç sayısı: 322
  En sık 10 ilaç:
İlaç Adı
AUGMENTIN BID 1000 MG.14 FILM TB.                 10
GERALGINE-K 500 MG/30 MG/10 MG 20 TABLET           9
PANTO 40 MG.28 TABLET                              8
MAJEZIK % 0.25 ORAL SPREY, COZELTI 30 ML           6
ARVELES 25 MG.20 FILM TABLET                       6
D-COLEFOR 20.000 IU YUMUSAK KAPSUL (12 KAPSUL)     5
CATAFLAM 50 MG.20 DRAJE                            5
FEMARA 2.5 MG 30 FTB                               5
ENFLUVIR 75 MG 10 KAP                              4
ONZYD 8 MG AGIZDA DAGILAN 10 TB                    4

  NSAİİ grubundan ilaç: 19 kayıt
  Kemoterapi grubundan ilaç: 0 kayıt

  Tüm kolonlar: ['HASTA_ID', 'SIKAYETNO', 'SUBEKODU', 'RECETE_TARIH', 'RF_EPISODE', 'İlaç Adı', 'Sıklık X Doz', 'VERİLİS_YOLU', 'Gün']

--- checkup_rec (500 satır) ---
  İlaç adı kolonu: İlaç Adı
  Benzersiz ilaç sayısı: 244
  En sık 10 ilaç:
İlaç Adı


---
## 📝 BÖLÜM 11: Metin Sütunlarının Zenginliği (NLP Potansiyeli)

In [27]:
print('=== METİN SÜTUNLARI (NLP İÇİN) ===\n')

TEXT_KEYWORDS = ['YAKINMA','ÖYKÜ','OYKÜ','MUAYENE','TEDAVİ','TEDAVI',
                 'NOT','OZGECMIS','ÖZGEÇMİŞ','NOTE','SUMMARY','EPIKRIZ',
                 'KONTROL','SIKAYETLERI']

for key in [k for k in dfs if 'ana' in k]:
    df = dfs[key]
    print(f'--- {key} ---')
    
    text_cols = [c for c in df.columns 
                 if any(kw in c.upper() for kw in TEXT_KEYWORDS)]
    
    if text_cols:
        for col in text_cols:
            dolu    = df[col].notna().sum()
            null_p  = df[col].isnull().mean() * 100
            avg_len = df[col].dropna().str.len().mean() if df[col].notna().any() else 0
            print(f'  {col:<35} {dolu:>4}/{len(df)} dolu (%{null_p:.0f} boş)  ort. {avg_len:.0f} karakter')
            if df[col].notna().any():
                sample = str(df[col].dropna().iloc[0])[:80]
                print(f'    Örnek: "{sample}"')
    else:
        print('  Metin sütunu bulunamadı')
    print()

=== METİN SÜTUNLARI (NLP İÇİN) ===

--- cancer_ana ---
  YAKINMA                              181/500 dolu (%64 boş)  ort. 50 karakter
    Örnek: "BOĞAZ AĞRISI ÖKSÜRÜK,"
  ÖYKÜ                                 181/500 dolu (%64 boş)  ort. 168 karakter
    Örnek: " "
  YAKINMA_BASLANGIC_ZAMANI             134/500 dolu (%73 boş)  ort. 19 karakter
    Örnek: "2016-10-07 12:00:49"
  Muayene Notu                         482/500 dolu (%4 boş)  ort. 110 karakter
    Örnek: "aktıf sınovıtı yok"
  Kontrol Notu                         197/500 dolu (%61 boş)  ort. 111 karakter
    Örnek: "10.07.2012-NÖROŞİRÜRJİ MUAYENESİ ÖNERİLDİ"
  Tedavi Notu                          375/500 dolu (%25 boş)  ort. 47 karakter
    Örnek: "off med takıp"
  Özgeçmiş Notu                        133/500 dolu (%73 boş)  ort. 50 karakter
    Örnek: "böbrek tbcGeçirdiği Hastalıklar: Evet"
  Soygeçmiş Notu                        90/500 dolu (%82 boş)  ort. 23 karakter
    Örnek: "adayıda ac ca"

--- checkup_ana ---
  YAKIN

---
## ✅ BÖLÜM 12: Özet Karar Tablosu

In [28]:
print('=== ÖZET: HANGİ KOHORTLA DEVAM? ===\n')
print('Bu hücreyi çalıştırdıktan sonra gördüklerinize göre değerlendirin:\n')

summary = []
for key, df in dfs.items():
    null_avg = df.isnull().mean().mean() * 100
    
    # Metin zenginliği
    TEXT_KEYWORDS = ['YAKINMA','ÖYKÜ','MUAYENE','NOT','OZGECMIS']
    text_cols = [c for c in df.columns 
                 if any(kw in c.upper() for kw in TEXT_KEYWORDS)]
    text_richness = len(text_cols)
    
    summary.append({
        'Tablo': key,
        'Satır': len(df),
        'Sütun': df.shape[1],
        'Ort. Boş%': f'{null_avg:.0f}%',
        'Metin Kol.': text_richness,
        'Durum': '✅ Analiz edilebilir' if null_avg < 50 else '⚠️  Çok boş'
    })

print(pd.DataFrame(summary).to_string(index=False))

print('''
─────────────────────────────────────────────────────
SONRAKI ADIM KARARI:
─────────────────────────────────────────────────────
A) Kolonlar 3 kohortda aynıysa   → Hepsini birleştir, kohort = yeni özellik
B) Cancer'da onkoloji kolonu varsa → Cancer'a özel Sentinel modülü ekle  
C) Check-Up en temizse           → Demo için Check-Up'ı öne çıkar
D) Ex = taburcu hastaysa         → Readmission modeli için ideal target
─────────────────────────────────────────────────────
''')

=== ÖZET: HANGİ KOHORTLA DEVAM? ===

Bu hücreyi çalıştırdıktan sonra gördüklerinize göre değerlendirin:

      Tablo  Satır  Sütun Ort. Boş%  Metin Kol.               Durum
 cancer_ana    500     59       59%           8         ⚠️  Çok boş
 cancer_lab    500      7        7%           0 ✅ Analiz edilebilir
 cancer_rec    500      9        1%           0 ✅ Analiz edilebilir
checkup_ana    500     61       57%           8         ⚠️  Çok boş
checkup_lab    500      7        8%           0 ✅ Analiz edilebilir
checkup_rec    500      9        2%           0 ✅ Analiz edilebilir
     ex_ana    500     59       68%           8         ⚠️  Çok boş
     ex_lab    500      7        6%           0 ✅ Analiz edilebilir
     ex_rec    500      9        7%           0 ✅ Analiz edilebilir

─────────────────────────────────────────────────────
SONRAKI ADIM KARARI:
─────────────────────────────────────────────────────
A) Kolonlar 3 kohortda aynıysa   → Hepsini birleştir, kohort = yeni özellik
B) Cancer

In [29]:
NOTLARIM = """
Cancer Ana  → 59 sütun | 6 hasta | HASTA_ID anonim | tarihler string format (dönüşüm gerekmez)
              ICD: J/E/M/R ağırlıklı, C kodu yok ama TUM_EPS_TANILAR'da %100 C50 (geçmişte meme ca)
              Metin: Muayene Notu %96 dolu (en zengin), YAKINMA/ÖYKÜ %64 boş
              Onkoloji'ye özgü sütun YOK — kolonlar CheckUp/Ex ile neredeyse aynı
              FEMARA + ONZYD reçetede → hormon tedavisi aşamasında kanser hastaları
              KARAR: Demo için ek senaryo (FEMARA + böbrek kontrendikasyon)

Cancer Lab  → 7 sütun | HASTA_ID ile join | tarihler string
              SADECE 6 benzersiz test (pH, BUN, Hemoglobin, Tokso, CMV, CK-MB) — çok kısıtlı
              Referans değerleri %98 dolu — anormal tespiti yapılabilir
              KARAR: Sentinel'e dahil ama CheckUp Lab kadar zengin değil

Cancer Rec  → 9 sütun | RF_EPISODE ile join | tüm kolonlar dolu
              322 benzersiz ilaç | NSAİİ: 19 kayıt | Kemoterapi: 0
              FEMARA (meme ca) + ONZYD (bulantı) + ARVELES (NSAİİ) — kontrendikasyon senaryosu hazır
              KARAR: Demo senaryosu için kullan (ARVELES yazılıyor → geçmişte C50 → UYARI)

CheckUp Ana → 61 sütun (2 ekstra: BASLANGIC_ZAMANI, RF_EPISODE2) | 11 hasta
              Metin: ÖYKÜ ort. 239 karakter — en zengin metin verisi bu kohortda
              Muayene Notu %93 dolu, YAKINMA/ÖYKÜ %50 dolu
              ICD: D22 (nevus/benign), Z (kontrol) ağırlıklı — gerçek check-up populasyonu
              KARAR: ANA DEMO KOHORTU — gizli risk tespiti için ideal

CheckUp Lab → 7 sütun | 72 dosya × 1M satır = 72M satır toplam
              128 benzersiz test — en zengin lab verisi
              Glukoz, Kreatinin, Hemoglobin, HbA1c, Kolesterol hepsi var
              REFMIN %21 / REFMAX %17 boş — büyük çoğunluk dolu
              KARAR: Sentinel'in LAB motoru bu veriyle beslenir, DuckDB zorunlu

CheckUp Rec → 9 sütun | tüm kolonlar dolu | 4 dosya
              Kardiyoloji + Diyabet ilaçları mevcut
              KARAR: İlaç-lab kontrendikasyon kontrolü için kullan

Ex Ana      → 59 sütun | 58 hasta | en fazla hasta çeşidi
              YAKINMA/ÖYKÜ %99 boş — klinik not girilmemiş
              Muayene Notu %97 dolu ama kısa
              ICD: F (Psikiyatri) ağırlıklı — psikiyatri + dahiliye karışık
              TUM_EPS_TANILAR zengin: E11 (Diyabet), kanser geçmişi var
              KARAR: Şimdilik kenara — metin çok boş, demo için zayıf

Ex Lab      → 7 sütun | 2 dosya | 73 benzersiz test
              TSH ağırlıklı (tiroid) — psikiyatri hastalarında tiroid takibi mantıklı
              Referans değerleri %88 dolu
              KARAR: Ek analiz için kullanılabilir, demo'da öncelik değil

Ex Rec      → 9 sütun | RECETE_TARIH %57 boş — veri kalitesi düşük
              289 benzersiz ilaç | psikiyatri ilaçları ağırlıklı (Seroquel vb.)
              KARAR: Kenara — tarih eksikliği ciddi sorun
"""

print(NOTLARIM)


Cancer Ana  → 59 sütun | 6 hasta | HASTA_ID anonim | tarihler string format (dönüşüm gerekmez)
              ICD: J/E/M/R ağırlıklı, C kodu yok ama TUM_EPS_TANILAR'da %100 C50 (geçmişte meme ca)
              Metin: Muayene Notu %96 dolu (en zengin), YAKINMA/ÖYKÜ %64 boş
              Onkoloji'ye özgü sütun YOK — kolonlar CheckUp/Ex ile neredeyse aynı
              FEMARA + ONZYD reçetede → hormon tedavisi aşamasında kanser hastaları
              KARAR: Demo için ek senaryo (FEMARA + böbrek kontrendikasyon)

Cancer Lab  → 7 sütun | HASTA_ID ile join | tarihler string
              SADECE 6 benzersiz test (pH, BUN, Hemoglobin, Tokso, CMV, CK-MB) — çok kısıtlı
              Referans değerleri %98 dolu — anormal tespiti yapılabilir
              KARAR: Sentinel'e dahil ama CheckUp Lab kadar zengin değil

Cancer Rec  → 9 sütun | RF_EPISODE ile join | tüm kolonlar dolu
              322 benzersiz ilaç | NSAİİ: 19 kayıt | Kemoterapi: 0
              FEMARA (meme ca) + ONZYD (bulantı) + ARV